In [ ]:
import pandas as pd
import time
from google import genai
from google.genai import types
from pydantic import BaseModel, Field


GEMINI_API_KEY = "REMEMBER to CHANGE with YOUR GEMINI API"


https://aistudio.google.com/api-keys

Get yout API here

In [ ]:


client = genai.Client(api_key=GEMINI_API_KEY)

# Define the exact JSON structure for a SINGLE ROW (combining all 4 columns).
class CleanedReviewRow(BaseModel):
    original_id: int = Field(description="The exact ID provided in the prompt mapping")
    summary: str = Field(description="Cleaned and noise-filtered summary. Empty string if original was meaningless.")
    advice_to_management: str = Field(description="Cleaned advice. Empty string if meaningless.")
    review_pros: str = Field(description="Cleaned pros. Empty string if meaningless.")
    review_cons: str = Field(description="Cleaned cons. Empty string if meaningless.")
    is_global_noise: bool = Field(description="True ONLY if ALL 4 fields combined are completely meaningless spam (e.g., just 'good good good' everywhere)")

# Define the structure for the BATCH (an array of rows).
class BatchResponse(BaseModel):
    results: list[CleanedReviewRow]

def process_batch_multi_col(batch_df: pd.DataFrame, target_columns: list, model_name="gemini-2.5-flash"):
    """
    Processes a chunk of the dataframe, feeding all specified columns per row 
    into a single API call to preserve context and minimize quota usage.
    """
    prompt = (
        "You are a data cleaning pipeline. Process the following employee reviews. "
        "If the language is not in English, please translate into English."
        "Fix severe grammatical errors, remove completely meaningless filler (like 'not bad', 'NA'), "
        "but preserve the core sentiment and complaints. If a specific field is meaningless, return an empty string for it.\n\n"
    )
    
    # Pack all 4 columns into a cohesive block for each row
    for index, row in batch_df.iterrows():
        prompt += f"--- RECORD ID: {index} ---\n"
        for col in target_columns:
            prompt += f"[{col}]: {row[col]}\n"
        prompt += "\n"
        
    try:
        # Force the model to return the specific multi-column Pydantic schema
        response = client.models.generate_content(
            model=model_name,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=BatchResponse,
                temperature=0.1 
            ),
        )
        return response.parsed 
    except Exception as e:
        print(f"Batch Failed for indices {batch_df.index.min()} to {batch_df.index.max()}: {e}")
        return None

# ---------------------------------------------------------
# 3. Execution (The Throttle Pipeline)
# ---------------------------------------------------------

if __name__ == "__main__":
    df = pd.read_csv("../data/processed/glassdoor_reviews_cleaned.csv")

    
    target_cols = ['summary', 'advice_to_management', 'review_pros', 'review_cons']
    
    # Ensure all target columns exist and handle NaNs securely before iteration
    for col in target_cols:
        df[col] = df[col].fillna("").astype(str)

    # 1000 rows / 50 batch size = exactly 20 requests total.
    BATCH_SIZE = 50
    all_cleaned_records = []

    for i in range(0, len(df), BATCH_SIZE):
        batch = df.iloc[i:i+BATCH_SIZE]
        print(f"Processing rows {i} to {i+BATCH_SIZE - 1}...")
        
        batch_result = process_batch_multi_col(batch, target_columns=target_cols)
        
        if batch_result and batch_result.results:
            all_cleaned_records.extend([item.model_dump() for item in batch_result.results])
        
        time.sleep(5) 

    final_cleaned_df = pd.DataFrame(all_cleaned_records)
    
    final_cleaned_df.set_index('original_id', inplace=True)
    df.update(final_cleaned_df)

Processing rows 0 to 49...
Processing rows 50 to 99...
Processing rows 100 to 149...
Processing rows 150 to 199...
Processing rows 200 to 249...
Processing rows 250 to 299...
Processing rows 300 to 349...
Processing rows 350 to 399...
Processing rows 400 to 449...
Processing rows 450 to 499...
Processing rows 500 to 549...
Processing rows 550 to 599...
Processing rows 600 to 649...
Processing rows 650 to 699...
Processing rows 700 to 749...
Processing rows 750 to 799...
Processing rows 800 to 849...
Processing rows 850 to 899...
Processing rows 900 to 949...
Processing rows 950 to 999...


In [ ]:
df.to_csv("cleaned.csv")